In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import geopandas as gpd

# -- Robustly locate the repository root (dir that contains 'src') --------
def _find_repo_root(start: Path) -> Path:
    """Walk up from *start* until we find a directory that contains 'src'."""
    for p in [start, *start.parents]:
        if (p / "src").is_dir():
            return p
    raise RuntimeError(f"Could not find a 'src' directory above {start}")

repo_root = _find_repo_root(Path.cwd())
_src = str(repo_root / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)
    
from hotelling.simulation.dense_log import DenseLog

print(f"repo_root : {repo_root}")
print(f"sys.path  : {_src}")

In [ ]:
RUN_DIR = repo_root / Path("results/runs/20260627_060916_1d51a4cc")
#RUN_DIR = repo_root / Path("results/strategic_runs/runs/20260704_183836_f8ac038d")

In [ ]:
# load .npz file
Q_table = np.load(RUN_DIR / "qtable.npz")

In [ ]:
dl = DenseLog.load(RUN_DIR)
_stride_df = max(1, dl._rows_written // (5_000_000 // max(1, len(dl.agent_ids))))
#_stride_df = 100
#df = dl.to_dataframe(step_slice=slice(0, dl._rows_written, _stride_df))

In [ ]:
# read parquet with supermarkets via gpd
supermarkets_gdf = gpd.read_parquet(repo_root / "data/processed/supermarkets.parquet")
supermarkets_gdf['id'] = supermarkets_gdf.index

bio_ids = supermarkets_gdf[supermarkets_gdf['chain_type'] == 'bio']['id'].tolist()
standard_ids = supermarkets_gdf[supermarkets_gdf['chain_type'] == 'standard']['id'].tolist()
discount_ids = supermarkets_gdf[supermarkets_gdf['chain_type'] == 'discount']['id'].tolist()

In [ ]:
df = dl.to_dataframe(step_slice=slice(0, dl._rows_written, _stride_df))

In [ ]:
m=30
# lin space of [41.5800, 60.6280] with 30 points
lin_space_D = np.linspace(41.5800, 60.6280, m)
lin_space_S = np.linspace(46.8000, 66.8249, m)
lin_space_B = np.linspace(53.4600, 74.5088, m)
# Step size of lin space
step_size_D = lin_space_D[1] - lin_space_D[0]
step_size_S = lin_space_S[1] - lin_space_S[0]
step_size_B = lin_space_B[1] - lin_space_B[0]
print(f"step_size_D: {step_size_D}")
print(f"step_size_S: {step_size_S}")
print(f"step_size_B: {step_size_B}")

In [ ]:
Q_table.get('q')

# Average the q tables across the first dimension
bio_q = np.mean(Q_table.get('q')[bio_ids], axis=0)
standard_q = np.mean(Q_table.get('q')[standard_ids], axis=0)
discount_q = np.mean(Q_table.get('q')[discount_ids], axis=0)

q_names = ['bio_q', 'standard_q', 'discount_q']

average_q = np.mean(Q_table.get('q'), axis=0)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
if True:
    for q, name in zip([bio_q, standard_q, discount_q], q_names):
        normalized_q = q / np.max(q)
        normalized_q[normalized_q == 0] = np.nan  # Set zeros to NaN for better visualization
        
        plt.figure(figsize=(20, 20), dpi=200)
        sns.heatmap(normalized_q, cmap="viridis")
        plt.title(f"{name} (average Q-table for {name.split('_')[0]} supermarkets)")
        plt.xlabel("Actions")
        plt.ylabel("States")
        plt.savefig(repo_root / 'report' / "figures" / f"{name}_heatmap.png", bbox_inches='tight', transparent=True, dpi=300)
        plt.show()
    for q, name in zip([bio_q, standard_q], q_names[:2]):
        normalized_q = q / np.max(q)
        normalized_q[normalized_q == 0] = np.nan  # Set zeros to NaN for better visualization
        d_norm_q = discount_q / np.max(discount_q)
        
        diff_q = normalized_q - d_norm_q
        
        plt.figure(figsize=(20, 20), dpi=200)
        sns.heatmap(diff_q, cmap="RdYlGn", center=0)
        plt.title(f"Difference between {name} and discount supermarkets (average Q-table)")
        plt.xlabel("Actions")
        plt.ylabel("States")
        plt.show()


In [ ]:
# choose random int 
random_int = np.random.randint(0, len(Q_table.get('q'))-1)

random_int = 434

q_mat = Q_table.get('q')[random_int]
normalized_q_mat = q_mat / np.max(q_mat)
normalized_q_mat[normalized_q_mat == 0] = np.nan  # Set zeros to NaN for better visualization
# plot matrix as heatmap

plt.figure(figsize=(20, 20), dpi=200)
sns.heatmap(normalized_q_mat, cmap="viridis")
plt.title(f"Q-table ({random_int})")
plt.xlabel("Actions")
plt.ylabel("States")
plt.show()

In [ ]:
# Replace all 0s by nan in the normalized tables
average_q_norm = average_q / np.max(average_q)
average_q_norm[average_q_norm == 0] = np.nan

In [ ]:
# plot matrix as heatmap
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(20, 20), dpi=200)
sns.heatmap(average_q_norm, cmap="viridis")
plt.title("Q-table (ave)")
plt.xlabel("Actions")
plt.ylabel("States")
plt.show()

In [ ]:
# Replace all 0s by nan in the normalized tables
average_q_norm[average_q_norm < 0.3] = np.nan

In [ ]:
# plot matrix as heatmap
import matplotlib.pyplot as plt
import seaborn as sns
sns.heatmap(average_q_norm, cmap="viridis")
plt.title("Q-table (first layer)")
plt.xlabel("Actions")
plt.ylabel("States")
plt.show()